## Post-processing COMSOL solution

In [7]:
import time
from pathlib import Path
from typing import Any
import dataclasses
import shutil
import numpy as np
from scipy.spatial import KDTree
import pyvista as pv

#%%
# Set file locations
case_name = "case1"
mesh_name = "coil_box_named.msh"

mesh_path = "../../meshes/"
output_path = f"../../output/{case_name}/{case_name}_comsol"
mesh_file_path = mesh_path + mesh_name

In [8]:
sim_data = pv.read(output_path + "_export" + ".vtu")

# Extract data for the cells corresponding to the coil.
sim_data = sim_data.extract_values(values=1, scalars="Material_settings", preference="point")

In [9]:
elec_pot_unordered = sim_data["Electric_potential"]
curr_dens_unordered_X = sim_data["Current_density,_X-component"][:, np.newaxis]
curr_dens_unordered_Y = sim_data["Current_density,_Y-component"][:, np.newaxis]
curr_dens_unordered_Z = sim_data["Current_density,_Z-component"][:, np.newaxis]

sol1_unordered = elec_pot_unordered
sol2_unordered = np.concatenate((curr_dens_unordered_X, curr_dens_unordered_Y, curr_dens_unordered_Z), axis=1)
# print(sol1_unordered.shape)
# print(sol2_unordered.shape)
# print(attribute.shape)

points_unordered = sim_data.points
res = pv.read(mesh_file_path)
points = res.points
tree2 = KDTree(points_unordered)

sol1 = np.zeros(len(points))
sol2 = np.zeros((len(points), 3))

for i in range(len(points)):
    _, index = tree2.query(points[i, :], distance_upper_bound=1e-9)
    if index == tree2.n:
        continue
    else:
        sol1[i] = sol1_unordered[index]
        sol2[i, :] = sol2_unordered[index]

res["electric_potential"] = sol1
res["current_density"] = sol2

res.save(output_path + ".vtu")